In [1]:
!pip install ldpc
!pip install bposd
!pip install mqt.qecc

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 1.3 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.1/343.1 kB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 1.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.4/618.4 kB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.3 MB/s eta 0:00:0000:0100:010m


In [2]:
from ldpc.codes import rep_code
from bposd.hgp import hgp

construct surface code based on rep_code

In [34]:
h = rep_code(3)
surface_code = hgp(h1=h, h2=h, compute_distance=True)
surface_code.test()

print(surface_code.hz)
print(surface_code.N)
1, 0, 0, 1, 0, 0, 1,

<Unnamed CSS code>, (2,4)-[[13,1,3]]
 -Block dimensions: Pass
 -PCMs commute hz@hx.T==0: Pass
 -PCMs commute hx@hz.T==0: Pass
 -lx \in ker{hz} AND lz \in ker{hx}: Pass
 -lx and lz anticommute: Pass
 -<Unnamed CSS code> is a valid CSS code w/ params (2,4)-[[13,1,3]]
[[1 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 1 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1]]
13


(1, 0, 0, 1, 0, 0, 1)

In [8]:
import numpy as np
hz_trans= np.array(
       [[1,  0, 0,  0, 0,  0, 1,0,0, 1, 0, 0, 0],
       [0,  1, 0,  0, 0,  0, 1,0,0, 0, 1, 0, 0],
       [0,  0, 1,  0, 0,  0, 0,1,0, 1, 0, 1, 0],
       [0,  0, 0,  1, 0,  0, 0,1,0, 0, 1, 0, 1],
       [0,  0, 0,  0, 1,  0, 0,0,1, 0, 0, 1, 0],
       [0,  0, 0,  0, 0,  1, 0,0,1, 0, 0, 0, 1]]
       )


initialize decoder
+ BP+OSD
+ UFDecoder

In [5]:
import numpy as np
from ldpc import bposd_decoder
from mqt.qecc import *  # UFDecoder

p = 0.05  # 错误率

# BP+OSD
bposd_decoder = bposd_decoder(
    surface_code.hz,
    error_rate=p,
    channel_probs=[None],
    max_iter=surface_code.N,
    bp_method="ms",
    ms_scaling_factor=0,
    osd_method="osd_cs",
    osd_order=7,
)

# UFDecoder
code = Code(surface_code.hx, surface_code.hz)
uf_decoder = UFHeuristic()
uf_decoder.set_code(code)

decode and calculate success rate
+ BP+OSD
+ UFDecoder (mqt-qecc)
+ Our method

In [6]:
# 上述surface_code.hz化简后的B
B = np.array(
    [
        [1,0,0, 1, 0, 0, 0],
        [1,0,0, 0, 1, 0, 0],
        [0,1,0, 1, 0, 1, 0],
        [0,1,0, 0, 1, 0, 1],
        [0,0,1, 0, 0, 1, 0],
        [0,0,1, 0, 0, 0, 1],
    ]
)

H_X = surface_code.hz

weights = [
    np.log((1 - p) / p) for i in range(H_X.shape[1])
]  # 初始每个qubit的对数似然比

W_f = weights[: H_X.shape[0]]
W_g = weights[H_X.shape[0] :]

W_f_B = np.dot(W_f, B)  # W_f * B
W_g_B_W_f = W_f_B + W_g  # W_f * B + W_g
# print(f"W_g_B_W_f = {W_g_B_W_f}")

g = np.where(
    W_g_B_W_f > 0,
    0,
    np.where(W_g_B_W_f < 0, 1, np.random.randint(0, 2, size=W_g_B_W_f.shape)),
)
# print(f"g = {g}")

B_g = np.dot(B, g)  # B * g
print(f"B_g = {B_g}")

B_g = [0 0 0 0 0 0]


In [49]:
surface_code.lz

array([[1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]])

In [52]:
num_trials = 1000

bposd_num_success = 0
uf_num_success = 0
our_num_success = 0

for i in range(num_trials):

    # generate error
    error = np.zeros(surface_code.N).astype(int)
    for q in range(surface_code.N):
        if np.random.rand() < 0.05:
            error[q] = 1
    # num_errors = np.random.randint(1, surface_code.N // 2)  # 随机选择1到N//2个错误位置
    # error_indices = np.random.choice(surface_code.N, num_errors, replace=False)
    # error[error_indices] = 1  # 在随机选中的量子比特上设置错误
    # # print(f"Trial {i + 1}, error = {error}")

    # obtain syndrome
    syndrome = surface_code.hz @ error % 2
    # print(f"Trial {i + 1}, syndrome = {syndrome}")

    """Decode"""
    # 1. BP+OSD
    bposd_decoder.decode(syndrome)

    # 2. UFDecoder
    uf_decoder.decode(syndrome)
    uf_result = np.array(uf_decoder.result.estimate).astype(int)

    # 3. Our Decoder
    syndrome_copy = syndrome.copy()
    # # syndrome_copy[0] = syndrome_copy[0] + syndrome_copy[1] * (-1)
    # # syndrome_copy[2] = syndrome_copy[2] + syndrome_copy[3] * (-1)
    # # syndrome_copy[4] = syndrome_copy[4] + syndrome_copy[5] * (-1)
    # trans_syndrome = np.where(syndrome_copy == -1, 1, syndrome_copy)
    f = (np.dot(B, g) + syndrome_copy)%2
    our_result = np.hstack((f, g))
    # print(((hz_trans @ our_result)%2 == syndrome).all())
    # print(f"Trial {i + 1}, bposd result = {bposd_decoder.osdw_decoding}")
    # print(f"Trial {i + 1}, uf result = {uf_result}")
    bposd_result =  bposd_decoder.osdw_decoding
    
   
    # calculate success rate for each decoder
    bposd_residual_error = (bposd_decoder.osdw_decoding + error) % 2
    bpflag = (surface_code.lz @ bposd_residual_error % 2).any()
    if bpflag == 0:
        bposd_num_success += 1

    uf_residual_error = (uf_result + error) % 2
    flag = (surface_code.lz @ uf_residual_error % 2).any()
    if flag == 0:
        uf_num_success += 1
        
    trans_results = np.zeros_like(our_result,dtype=int)
    trans_results[0] = our_result[0]
    trans_results[1] = our_result[6]
    trans_results[2] = our_result[1]
    trans_results[3] = our_result[2]
    trans_results[4] = our_result[7]
    trans_results[5] = our_result[3]
    trans_results[6] = our_result[4]
    trans_results[7] = our_result[8]
    trans_results[8] = our_result[5]
    trans_results[9:-1] = our_result[9:-1]
    assert ((surface_code.hz @ trans_results)%2 == syndrome).all()
    # assert ((surface_code.hz @ error)%2 == syndrome).all()

    # print((surface_code.hz @ our_result)%2 ,syndrome)
    our_residual_error = (trans_results + error) % 2
    flag = (surface_code.lz @ our_residual_error % 2).any()
    if flag == 0:
        our_num_success += 1
    else:
        # print(surface_code.lz *trans_results,surface_code.lz *error)
        # print(trans_results,error)
        if bpflag==0:
            print([bposd_result[1],bposd_result[4],bposd_result[7]]+list(bposd_result[9:13]))
        
        pass

bposd_success_rate = bposd_num_success / num_trials
uf_success_rate = uf_num_success / num_trials
our_success_rate = our_num_success / num_trials
print(f"\nTotal trials: {num_trials}")
print(f"BP+OSD Success rate: {bposd_success_rate * 100:.2f}%")
print(f"UF Success rate: {uf_success_rate * 100:.2f}%")
print(f"Our Success rate: {our_success_rate * 100:.2f}%")

[1, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 0, 1, 1, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0]
[1, 0, 0, 

In [44]:
import numpy as np

# 高斯消元法（mod 2）
def gauss_elimination_mod2(A, b):
    n = len(A)
    
    # 合并矩阵 A 和列向量 b，形成增广矩阵
    Augmented = A.copy()
    
    for i in range(n):
        # 寻找主元
        if Augmented[i, i] == 0:
            # 如果主元为0，寻找下面一行有1的列交换
            print(i,i)
            for j in range(i, n):
                if Augmented[i,j] == 1:
                    print(j,i)
                    Augmented[[i, j]] = Augmented[[j, i]]
                    break
    print(Augmented)
    for i in range(n):
        # 对主元所在行进行消元
        if Augmented[i, i] == 1:
            for j in range(i+1, n):
                if Augmented[j, i] == 1:
                    Augmented[j] ^= Augmented[i]
    # print(Augmented)
    # # 反向消元
    # for i in range(n-1, -1, -1):
    #     for j in range(i-1, -1, -1):
    #         if Augmented[j, i] == 1:
    #             Augmented[j] ^= Augmented[i]
    
    # # 提取解向量
    # solution = Augmented[:, -1]
    return Augmented

# 示例
A = surface_code.hz

b = np.zeros(6,dtype=int)

# 解决线性方程组
Agauss = gauss_elimination_mod2(A, b)
print(Agauss)

2 2
3 3
4 4
5 5
[[1 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 1 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1]]
[[1 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 1 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1]]


Perform Gaussian Elimination on parity check matrix Hz

下面的高斯消元代码结果不对，如果矩阵满秩，一定能化成[I, B]

In [41]:
import numpy as np


def gaussian_elimination(H, s):

    m, n = H.shape
    H_prime = H.copy()
    s_prime = s.copy()

    # Start Gaussian elimination
    for i in range(m):
        # Ensure the pivot element is 1
        if H_prime[i, i] == 0:
            # Swap rows if needed to make H_prime[i, i] == 1
            for j in range(i + 1, m):
                if H_prime[j, i] == 1:
                    H_prime[[i, j]] = H_prime[[j, i]]
                    s_prime[[i, j]] = s_prime[[j, i]]
                    break

        # Eliminate entries below the pivot
        for j in range(i + 1, m):
            if H_prime[j, i] == 1:
                H_prime[j] = (H_prime[j] + H_prime[i]) % 2
                s_prime[j] = (s_prime[j] + s_prime[i]) % 2

    # Perform back substitution to get the final form [I | B]
    for i in range(m - 1, -1, -1):
        for j in range(i - 1, -1, -1):
            if H_prime[j, i] == 1:
                H_prime[j] = (H_prime[j] + H_prime[i]) % 2
                s_prime[j] = (s_prime[j] + s_prime[i]) % 2

    return H_prime, s_prime


# Example usage:
# Suppose H is a 3x6 matrix and s is a 3x1 syndrome vector
H = np.array([[1, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 1], [0, 1, 1, 1, 1, 0]], dtype=int)

s = np.array([1, 0, 1], dtype=int)

# Perform Gaussian elimination
H_prime, s_prime = gaussian_elimination(H, s)

print("Reduced matrix H':")
print(H_prime)
print("Adjusted syndrome vector s':")
print(s_prime)

"""
1 0 0 0 0 1
0 1 0 1 0 0
0 0 1 0 1 0
"""

Reduced matrix H':
[[1 0 1 0 1 1]
 [0 1 1 1 1 0]
 [0 0 0 0 0 0]]
Adjusted syndrome vector s':
[0 1 0]


'\n1 0 0 0 0 1\n0 1 0 1 0 0\n0 0 1 0 1 0\n'

验证矩阵是否满秩

In [42]:
import numpy as np


def is_full_rank(H_X):
    """
    判断校验矩阵 H_X 是否为满秩 (Full Rank)
    """
    rank = np.linalg.matrix_rank(H_X)  # 计算矩阵的秩
    print(f"rank = {rank}")
    num_rows = H_X.shape[0]  # 计算行数
    return rank == num_rows  # 满秩返回 True，否则 False


# 示例矩阵
H_X = surface_code.hz
# H_X = H

print("H_X 是否满秩:", is_full_rank(H_X))
print(H_X)

rank = 6
H_X 是否满秩: True
[[1 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 1 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1]]
